# Transcript Builder

In [1]:
%load_ext autoreload
%autoreload
import os

from exports.builders import MAFBuilder, TranscriptBuilder
from exports.builders.utils import struct_select, transcript_id_udf, all_effects_udf, extract_rows_udf

from config import TestConfig

conf = TestConfig()

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

%cd /mnt/Projects/gdc-mutation-indexer

/mnt/Projects/gdc-mutation-indexer


In [2]:
maf_df = MAFBuilder(conf, sqlContext).build()

In [3]:
ann_df = maf_df.select(*struct_select('annotation.yml'))\
                                .drop_duplicates(['transcript_id'])

### Explode the transcripts_ids for each ssm_id

In [28]:
ssm_tran = maf_df.select('gene_id','symbol','empty','ssm_id', 'all_effects', 'canonical_transcript_id')\
                .withColumn('all_effects',extract_rows_udf()(col('all_effects')).alias('all_effects'))\
                .select('gene_id','symbol','empty','ssm_id', 'canonical_transcript_id', explode('all_effects').alias('all_effects'))\
                .withColumn('do_not_keep', all_effects_udf(0)(col('all_effects')))\
                .withColumn('consequence_type', all_effects_udf(1)(col('all_effects')))\
                .withColumn('aa_change', all_effects_udf(2)(col('all_effects')))\
                .withColumn('transcript_id', all_effects_udf(3)(col('all_effects')))\
                .withColumn('ref_seq_accession', all_effects_udf(4)(col('all_effects')))\
                .drop('all_effects')
               # .select('ssm_id','canonical_transcript_id','all_effects.*')
               #.select('ssm_id', explode('transcript_ids').alias('transcript_id'))
               #.drop_duplicates(['transcript_id', 'ssm_id'])

In [29]:
# Should be 7
#print ssm_tran.where(ssm_tran.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').count()
#ssm_tran.where(ssm_tran.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').show(10)

In [30]:
ssm_tran.printSchema()

root
 |-- gene_id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- empty: string (nullable = true)
 |-- ssm_id: string (nullable = true)
 |-- canonical_transcript_id: string (nullable = true)
 |-- do_not_keep: string (nullable = true)
 |-- consequence_type: string (nullable = true)
 |-- aa_change: string (nullable = true)
 |-- transcript_id: string (nullable = true)
 |-- ref_seq_accession: string (nullable = true)



### Fast method (join gene later)

In [31]:
# Create transcript df 
tran_df = ssm_tran.select('gene_id', 'ssm_id', 'transcript_id', struct(*struct_select('transcript.yml')).alias('transcript'))

In [32]:
tran_df.printSchema()

root
 |-- gene_id: string (nullable = true)
 |-- ssm_id: string (nullable = true)
 |-- transcript_id: string (nullable = true)
 |-- transcript: struct (nullable = false)
 |    |-- is_canonical: string (nullable = true)
 |    |-- consequence_type: string (nullable = true)
 |    |-- aa_end: string (nullable = true)
 |    |-- gene_symbol: string (nullable = true)
 |    |-- ref_seq_accession: string (nullable = true)
 |    |-- transcript_id: string (nullable = true)
 |    |-- aa_start: string (nullable = true)
 |    |-- aa_change: string (nullable = true)



In [33]:
tran_ann = tran_df.join(ann_df, on='transcript_id', how='left')\
            .select('transcript_id', 'gene_id',
                    struct(ann_df.columns).alias('annotation'))

In [18]:
# Should be 7
#print tran_df.where(tran_df.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').count()
#tran_df.where(tran_df.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').show(10)

In [19]:
# Should be 7
#print tran_ann.where(tran_ann.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').count()
#tran_ann.where(tran_ann.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').show(10)

In [37]:
gene_df = maf_df.select(*struct_select('gene.yml', ignore=['transcripts']))\
                .drop('transcripts')\
                .drop('description')\
                .drop('canonical_transcript_length_genomic')\
                .drop('canonical_transcript_length_cds')\
                .drop('gene_strand')\
                .select('gene_id', struct(col('*')).alias('gene'))

In [53]:
#tran_df.printSchema()

In [61]:
tran_df = tran_ann.join(gene_df, on='gene_id')\
                        .drop('gene_id')\
                        .join(ssm_tran, on='transcript_id')\
                        .drop('empty')\
                        .drop('symbol')\
                        .drop('gene_id')\
                        .select('ssm_id', struct(
                                            struct('*')
                                            .alias('transcript'))
                                          .alias('consequence'))
            
#tran_df = tran_ann.join(ssm_tran, on='transcript_id')\
#        .select('ssm_id', struct(struct('*').alias('transcript')).alias('transcript'))
#tran_df.explain()

tran_df = tran_df.groupby('ssm_id').agg(collect_list('consequence').alias('consequence'))

# Should be 1
#print tran_df.where(tran_df.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').count()
# Should be 7
#tran_df.where(tran_df.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').select(size('consequence')).show()
#tran_df.printSchema()

In [63]:
tran_df.printSchema()

root
 |-- ssm_id: string (nullable = true)
 |-- consequence: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- transcript: struct (nullable = false)
 |    |    |    |-- transcript_id: string (nullable = true)
 |    |    |    |-- annotation: struct (nullable = false)
 |    |    |    |    |-- impact: string (nullable = true)
 |    |    |    |    |-- amino_acids: string (nullable = true)
 |    |    |    |    |-- existing_variation: string (nullable = true)
 |    |    |    |    |-- sift: string (nullable = true)
 |    |    |    |    |-- pubmed: string (nullable = true)
 |    |    |    |    |-- dbsnp_val_status: string (nullable = true)
 |    |    |    |    |-- ccds: string (nullable = true)
 |    |    |    |    |-- cdna_position: string (nullable = true)
 |    |    |    |    |-- cds_start: string (nullable = true)
 |    |    |    |    |-- hgvsp: string (nullable = true)
 |    |    |    |    |-- ensp: string (nullable = true)
 |    |    |    |    |-- dbs

In [20]:
#ssm_tran.select('transcript_ids','transcript_ids_2').limit(3).show(3, False)

In [21]:
#[t['transcript_id'] for t in ssm_tran.where(ssm_tran.ssm_id == '683cd056-cec3-59f4-9751-ef6453be5a9d').collect()]

In [22]:
#tran_df = sqlContext.read.json(conf.gene_model_file)
#tran_df.where(tran_df._gene_id=='ENSG00000013810').select('transcripts.id').collect()

### Slow method

In [45]:
## Slow method
tran_df = maf_df.select('transcript_id', *struct_select('gene.yml', ignore=['transcripts']))\
                 .select(col('*'), explode('transcripts').alias('transcript'))\
                 .drop('transcripts')
# Move gene under transcript
tran_df = tran_df.select('transcript_id', struct(*[c for c in tran_df.columns if c not in ['transcript_id','transcript']]).alias('gene'))\
                    .drop('domains')

In [46]:
#tran_df.printSchema()

In [47]:
tran_ann = tran_df.join(ann_df, on='transcript_id')\
                    .select('transcript_id', 'gene',struct(ann_df.columns).alias('annotation'))

In [57]:
tran_df = tran_ann.join(ssm_tran, on='transcript_id')#.select('ssm_id', struct('*').alias('transcript'))

In [59]:
tran_df.count()

253

In [60]:
#tran_ann.printSchema()
# Should be 7
tran_df.where(tran_df.ssm_id == '1a191926-2c54-539a-817d-6196d105bb38').count()

16